In [3]:
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
import pandas as pd
import numpy as np
import re
import os
from scipy.stats import kurtosis, skew
from datetime import datetime
import glob

In [13]:
import glob
import os

# 1. ระบุ Path (ลองเปลี่ยนจาก \ เป็น / ดูครับ)
folder_path = r'C:/Users/66851/Documents/MachineLearningProject/GCME-AI-Project2026/TWF/' # <-- แก้ไข Path ตรงนี้ให้เป๊ะ

# 2. ค้นหา
files = glob.glob(os.path.join(folder_path, '*.rtf'))

if len(files) > 0:
    print(f"✅ สำเร็จ! พบไฟล์ RTF {len(files)} ไฟล์")
    print("ตัวอย่างไฟล์ที่พบ:", files[0])
else:
    print("❌ ยังไม่พบไฟล์ครับ")
    print("Path ที่คุณกรอกคือ:", os.path.abspath(folder_path))

✅ สำเร็จ! พบไฟล์ RTF 1302 ไฟล์
ตัวอย่างไฟล์ที่พบ: C:/Users/66851/Documents/MachineLearningProject/GCME-AI-Project2026/TWF\0520P03-M1H_10-Dec-25_10-52-13.rtf


In [19]:
import pandas as pd
import numpy as np
import re
import os
import glob
from scipy.stats import kurtosis, skew
from datetime import datetime

def extract_features_safely(file_path):
    try:
        # 1. จัดการชื่อไฟล์และ Metadata
        basename = os.path.basename(file_path).replace('.rtf', '')
        parts = basename.split('_')
        if len(parts) < 2: return None
            
        machine_point = parts[0]
        date_str = parts[1]
        
        if '-' not in machine_point: return None
        machine, point = machine_point.split('-', 1)
        
        # แปลงวันที่ให้ตรงกับ Format ใน CSV (M/D/YYYY)
        try:
            dt_obj = datetime.strptime(date_str, '%d-%b-%y')
            csv_date = f"{dt_obj.month}/{dt_obj.day}/{dt_obj.year}"
        except: return None

        # 2. อ่านไฟล์และดึงสัญญาณ Amplitude
        with open(file_path, 'r', encoding='latin-1') as f:
            content = f.read()
        
        data_start = content.find('---------')
        if data_start == -1: return None
        
        raw_text = content[data_start:]
        numbers = re.findall(r'-?\d*\.\d+|-?\d+', str(raw_text))
        
        if len(numbers) < 2: return None
        
        # ดึง Amplitude (ตำแหน่งคี่)
        amplitudes = np.array([float(numbers[i]) for i in range(1, len(numbers), 2)])
        
        if len(amplitudes) == 0: return None

        # 3. คำนวณค่าสถิติ (Features)
        rms = np.sqrt(np.mean(amplitudes**2))
        peak = np.max(np.abs(amplitudes))
        kurt = kurtosis(amplitudes, fisher=False)
        skw = skew(amplitudes)
        crest = peak / rms if rms != 0 else 0
        
        return {
            'date': csv_date, 'machine': machine, 'point': point,
            'RMS': rms, 'Peak': peak, 'Kurtosis': kurt, 
            'Skewness': skw, 'Crest_Factor': crest
        }
    except:
        return None

# --- ส่วนประมวลผลหลัก ---

# 1. กำหนด Path ที่พบไฟล์แน่นอนแล้ว
folder_path = r'C:/Users/66851/Documents/MachineLearningProject/GCME-AI-Project2026/TWF/'

print("กำลังเริ่มสกัดข้อมูลจากไฟล์ RTF 1,302 ไฟล์... โปรดรอสักครู่")

all_rtf = glob.glob(os.path.join(folder_path, '*.rtf'))

# 2. เริ่มสกัดข้อมูล
extracted_list = []
for f in all_rtf:
    feat = extract_features_safely(f)
    if feat:
        extracted_list.append(feat)

df_features = pd.DataFrame(extracted_list)

if df_features.empty:
    print("❌ ไม่สามารถสกัดข้อมูลได้ กรุณาตรวจสอบรูปแบบเนื้อหาในไฟล์ RTF")
else:
    print(f"✅ สกัด Feature สำเร็จ: {len(df_features)} รายการ")

    # 3. โหลด CSV และทำการ Merge
    # ตรวจสอบว่าไฟล์ Bearing_Data_No_Zeros.csv อยู่ในโฟลเดอร์เดียวกับโค้ดนะครับ
    df_csv = pd.read_csv('Bearing_Data_No_Zeros.csv')

    # ล้างข้อมูล String เพื่อป้องกันการจับคู่พลาด
    for df in [df_csv, df_features]:
        df['machine'] = df['machine'].astype(str).str.strip().str.upper()
        df['point'] = df['point'].astype(str).str.strip().str.upper()
        df['date'] = df['date'].astype(str).str.strip()

    # 4. Merge ข้อมูล (Inner Join)
    df_combined = pd.merge(df_csv, df_features, on=['date', 'machine', 'point'], how='inner')

    print(f"✅ จับคู่ข้อมูล CSV กับ RTF สำเร็จ: {len(df_combined)} แถว")

    if not df_combined.empty:
        # 5. บันทึกเป็น Master Dataset
        df_combined.to_csv('Master_Dataset_for_ML.csv', index=False)
        print("🚀 สร้างไฟล์ 'Master_Dataset_for_ML.csv' เรียบร้อย! พร้อมทำ Machine Learning แล้วครับ")
    else:
        print("⚠️ จับคู่ไม่ได้เลย! กรุณาตรวจสอบว่า Date/Machine ใน CSV ตรงกับชื่อไฟล์ RTF หรือไม่")

กำลังเริ่มสกัดข้อมูลจากไฟล์ RTF 1,302 ไฟล์... โปรดรอสักครู่
✅ สกัด Feature สำเร็จ: 1302 รายการ
✅ จับคู่ข้อมูล CSV กับ RTF สำเร็จ: 33 แถว
🚀 สร้างไฟล์ 'Master_Dataset_for_ML.csv' เรียบร้อย! พร้อมทำ Machine Learning แล้วครับ


In [21]:
# เช็คว่าชื่อเครื่องจักรในไฟล์ RTF มีอยู่ใน CSV บ้างไหม
rtf_machines = df_features['machine'].unique()
csv_machines = df_csv['machine'].unique()

missing_machines = [m for m in rtf_machines if m not in csv_machines]
print(f"เครื่องจักรจาก RTF ที่หาไม่เจอใน CSV: {missing_machines[:10]}")

# เช็คว่าวันที่ใน RTF มีอยู่ใน CSV บ้างไหม
rtf_dates = df_features['date'].unique()
csv_dates = df_csv['date'].unique()

missing_dates = [d for d in rtf_dates if d not in csv_dates]
print(f"วันที่จาก RTF ที่หาไม่เจอใน CSV: {missing_dates[:10]}")

เครื่องจักรจาก RTF ที่หาไม่เจอใน CSV: ['AB', 'G', 'K', 'P', 'R']
วันที่จาก RTF ที่หาไม่เจอใน CSV: ['8/22/2024', '10/6/2025', '8/15/2025', '12/3/2025', '2/7/2022']


In [23]:
# 1. ดูรูปแบบวันที่และชื่อเครื่องใน CSV
print("--- ตัวอย่างข้อมูลใน CSV ---")
print(df_csv[['date', 'machine', 'point']].head())

# 2. ดูรูปแบบที่สกัดได้จากไฟล์ RTF (ก่อนการ Merge)
print("\n--- ตัวอย่างข้อมูลที่สกัดจากชื่อไฟล์ RTF ---")
print(df_features[['date', 'machine', 'point']].head())

--- ตัวอย่างข้อมูลใน CSV ---
        date machine point
0  6/27/2025  P-210S   M1A
1  6/27/2025  P-210S   M1V
2  6/27/2025  P-210S   M1H
3  6/27/2025  P-210S   M2A
4  6/27/2025  P-210S   M2V

--- ตัวอย่างข้อมูลที่สกัดจากชื่อไฟล์ RTF ---
         date  machine point
0  12/10/2025  0520P03   M1H
1   6/25/2025  0520P03   M1H
2  10/30/2025  0520P03   M1H
3  12/10/2025  0520P03   M1V
4   6/25/2025  0520P03   M1V


In [25]:
# ดูว่ามีเครื่องจักรชื่ออะไรบ้างที่ "มีอยู่ทั้งคู่" แต่ดันจับคู่ไม่ติด
machine_in_csv = set(df_csv['machine'].unique())
machine_in_rtf = set(df_features['machine'].unique())

common_machines = machine_in_csv.intersection(machine_in_rtf)
print(f"ชื่อเครื่องที่ตรงกันเป๊ะ: {common_machines}")

# เช็ควันที่
date_in_csv = set(df_csv['date'].unique())
date_in_rtf = set(df_features['date'].unique())

common_dates = date_in_csv.intersection(date_in_rtf)
print(f"วันที่ที่ตรงกันเป๊ะ: {common_dates}")

ชื่อเครื่องที่ตรงกันเป๊ะ: {'0520P03', '2200P039'}
วันที่ที่ตรงกันเป๊ะ: {'7/17/2025', '5/7/2025', '5/23/2025', '6/17/2025', '6/12/2025', '12/13/2025', '9/4/2025', '10/30/2025', '9/10/2025', '2/7/2025', '5/22/2025', '5/21/2025', '12/19/2025', '5/9/2025', '5/10/2025', '3/19/2025', '10/19/2023', '1/20/2025', '12/14/2025', '5/8/2025', '3/6/2025', '9/6/2024', '1/14/2026', '2/3/2025', '12/22/2025', '11/8/2024', '10/21/2025', '7/2/2024', '9/18/2025', '6/7/2024', '5/10/2024', '1/27/2025', '1/11/2025', '8/29/2025', '4/21/2025', '12/1/2025', '3/17/2025', '5/28/2025', '5/19/2025', '12/16/2024', '3/21/2025', '11/7/2025', '6/19/2025', '8/30/2024', '12/8/2025', '8/13/2025', '3/18/2025', '1/9/2025', '9/11/2025', '7/11/2025', '6/10/2024', '8/8/2025', '5/11/2025', '4/25/2025', '8/18/2025', '9/16/2025', '11/3/2025', '11/28/2025', '10/16/2025', '12/4/2025', '6/25/2025', '5/15/2025', '1/8/2025', '2/18/2019', '9/17/2025', '4/2/2025', '7/24/2025', '1/8/2026', '12/17/2024', '11/27/2025', '10/8/2025', '1/17/20

In [27]:
import pandas as pd
import numpy as np
import re
import os
import glob
from scipy.stats import kurtosis, skew
from datetime import datetime

def extract_features_improved(file_path):
    try:
        basename = os.path.basename(file_path).replace('.rtf', '')
        parts = basename.split('_')
        if len(parts) < 2: return None
            
        machine_point = parts[0]
        date_str = parts[1]
        
        # --- ปรับปรุงตรงนี้: แยก Machine และ Point ให้รองรับขีดกลางหลายตัว ---
        if '-' in machine_point:
            tmp_parts = machine_point.split('-')
            point = tmp_parts[-1]              # ตัวสุดท้ายคือ Point (เช่น M1H)
            machine = "-".join(tmp_parts[:-1]) # ที่เหลือข้างหน้าคือ Machine (เช่น P-210S)
        else:
            return None # หรือกำหนดค่า default ตามความเหมาะสม
        # -----------------------------------------------------------

        try:
            dt_obj = datetime.strptime(date_str, '%d-%b-%y')
            csv_date = f"{dt_obj.month}/{dt_obj.day}/{dt_obj.year}"
        except: return None

        with open(file_path, 'r', encoding='latin-1') as f:
            content = f.read()
        
        data_start = content.find('---------')
        if data_start == -1: return None
        
        raw_text = content[data_start:]
        numbers = re.findall(r'-?\d*\.\d+|-?\d+', str(raw_text))
        if len(numbers) < 2: return None
        
        amplitudes = np.array([float(numbers[i]) for i in range(1, len(numbers), 2)])
        if len(amplitudes) == 0: return None

        return {
            'date': csv_date, 'machine': machine, 'point': point,
            'RMS': np.sqrt(np.mean(amplitudes**2)), 
            'Peak': np.max(np.abs(amplitudes)),
            'Kurtosis': kurtosis(amplitudes, fisher=False),
            'Skewness': skew(amplitudes),
            'Crest_Factor': (np.max(np.abs(amplitudes)) / np.sqrt(np.mean(amplitudes**2))) if np.any(amplitudes) else 0
        }
    except:
        return None

# --- ส่วนประมวลผลหลัก ---
folder_path = r'C:/Users/66851/Documents/MachineLearningProject/GCME-AI-Project2026/TWF/'
all_rtf = glob.glob(os.path.join(folder_path, '*.rtf'))

extracted_list = []
for f in all_rtf:
    feat = extract_features_improved(f)
    if feat:
        extracted_list.append(feat)

df_features = pd.DataFrame(extracted_list)

# โหลด CSV และล้างช่องว่าง
df_csv = pd.read_csv('Bearing_Data_No_Zeros.csv')
for df in [df_csv, df_features]:
    df['machine'] = df['machine'].astype(str).str.strip().str.upper()
    df['point'] = df['point'].astype(str).str.strip().str.upper()
    df['date'] = df['date'].astype(str).str.strip()

# Merge อีกครั้ง
df_combined = pd.merge(df_csv, df_features, on=['date', 'machine', 'point'], how='inner')

print(f"✅ ครั้งนี้จับคู่ได้สำเร็จ: {len(df_combined)} แถว")
if not df_combined.empty:
    df_combined.to_csv('Master_Dataset_for_ML_v2.csv', index=False)
    print("🚀 สร้างไฟล์ 'Master_Dataset_for_ML_v2.csv' เรียบร้อย!")

✅ ครั้งนี้จับคู่ได้สำเร็จ: 994 แถว
🚀 สร้างไฟล์ 'Master_Dataset_for_ML_v2.csv' เรียบร้อย!


In [29]:
# เช็คชื่อเครื่องใน RTF ที่ยังไม่มีใน CSV
rtf_not_in_csv = set(df_features['machine'].unique()) - set(df_csv['machine'].unique())
print(f"ชื่อเครื่องใน RTF ที่ยังหาไม่เจอใน CSV (ลองเช็คดูว่าสะกดต่างกันไหม):")
print(sorted(list(rtf_not_in_csv))[:20])

ชื่อเครื่องใน RTF ที่ยังหาไม่เจอใน CSV (ลองเช็คดูว่าสะกดต่างกันไหม):
['G-2511A', 'G-315A', 'G-750A', 'G-751A', 'G-9107B', 'K-201CP01A', 'P-1014B', 'P-1014C', 'P-1200B', 'P-2013A']
